In [1]:
import os

from tensordev.kernel import HigherOrderKernel

os.environ.setdefault(
    "XLA_FLAGS",
    f"--xla_force_host_platform_device_count={os.cpu_count()}",
)

from jax import config

config.update("jax_enable_x64", True)

import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import jax.random as jr

%load_ext autoreload
%autoreload 1

from tensordev.kernel.free import free_kernel, FreeKernel
from tensordev.kernel.higher_order import higher_order_kernel


def sample_bm_paths(key, *, n_pairs, n_steps, dim, horizon=1.0):
    dt = horizon / n_steps
    kx, ky = jr.split(key)

    dX = jnp.sqrt(dt) * jr.normal(kx, (n_pairs, n_steps, dim), dtype=jnp.float64)
    dY = jnp.sqrt(dt) * jr.normal(ky, (n_pairs, n_steps, dim), dtype=jnp.float64)

    x0 = jnp.zeros((n_pairs, 1, dim), dtype=jnp.float64)
    y0 = jnp.zeros((n_pairs, 1, dim), dtype=jnp.float64)

    X = jnp.concatenate([x0, jnp.cumsum(dX, axis=1)], axis=1)
    Y = jnp.concatenate([y0, jnp.cumsum(dY, axis=1)], axis=1)

    return X, Y

In [ ]:
SEED = 240078029296
DIM = 2
N_FINE = 2 ** 12
DEGREES = [1, 2, 3, 4]
K_VALUES = list(range(4 + 2, 12))
N_PAIRS = 100
HORIZON = 1.0
NUM_DEVICES = jax.device_count()
CHUNK_SIZE = 25  # tune to fit your memory budget
DYO_FINE = 4  #4
DYO_HO = 7  #7

key = jr.PRNGKey(SEED)
X_paths, Y_paths = sample_bm_paths(
    key,
    n_pairs=N_PAIRS,
    n_steps=N_FINE,
    dim=DIM,
    horizon=HORIZON,
)

dX = (jnp.diff(X_paths, axis=1),)
dY = (jnp.diff(Y_paths, axis=1),)

_ref_kernel = FreeKernel(backend="scan", dyadic_order=DYO_FINE, num_devices=2, increment_input=True)
fine_vals = np.asarray(_ref_kernel(dX, dY, max_batch=CHUNK_SIZE), dtype=np.float64)

In [ ]:
from IPython.display import clear_output, display

AGGREGATION = "mean_log"  # "log_mean" or "mean_log"
EPS = 1e-300

mean_errors = {n: [] for n in DEGREES}
all_errors = {n: [] for n in DEGREES}

# Sweep order: coarse first = faster first, and also left-to-right in the plot
K_SWEEP = K_VALUES[::-1]
x_sweep = -np.log2(np.array([(2 ** k) / N_FINE for k in K_SWEEP], dtype=float))


def aggregate_errors(errs, mode="log_mean", eps=1e-300):
    errs = np.asarray(errs, dtype=float)
    if mode == "log_mean":
        return -np.log10(np.maximum(np.mean(errs), eps))
    elif mode == "mean_log":
        return np.mean(-np.log10(np.maximum(errs, eps)))
    else:
        raise ValueError("mode must be 'log_mean' or 'mean_log'")


def ylabel_for(mode):
    if mode == "log_mean":
        return r"$-\log_{10}(\mathrm{mean\ absolute\ error})$"
    elif mode == "mean_log":
        return r"$\mathrm{mean}\!\left[-\log_{10}(\mathrm{absolute\ error})\right]$"
    else:
        raise ValueError("mode must be 'log_mean' or 'mean_log'")


def render_progress():
    clear_output(wait=True)
    print(f"Aggregation mode: {AGGREGATION}")

    fig, ax = plt.subplots(figsize=(7.8, 4.8), constrained_layout=True)

    for n in DEGREES:
        if not mean_errors[n]:
            continue

        xcurr = x_sweep[:len(mean_errors[n])]
        ecurr = np.array(mean_errors[n], dtype=float)

        if AGGREGATION == "log_mean":
            yplot = -np.log10(np.maximum(ecurr, EPS))
        else:
            yplot = np.array([
                np.mean(-np.log10(np.maximum(errs, EPS)))
                for errs in all_errors[n]
            ], dtype=float)

        ax.plot(xcurr, yplot, marker="o", linewidth=2, markersize=5, label=f"{n}")

    ax.set_xlabel(r"$-\log_2(\mathrm{mesh\ size})$")
    ax.set_ylabel(ylabel_for(AGGREGATION))
    ax.set_title("Higher-order log-linear approximation of the signature kernel")
    ax.grid(True, alpha=0.3)
    ax.legend(
        title="degree",
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        borderaxespad=0.0,
        frameon=False,
    )

    display(fig)
    plt.close(fig)

    for n in DEGREES:
        if not mean_errors[n]:
            continue

        xcurr = x_sweep[:len(mean_errors[n])]
        print(f"degree {n}:")
        for i, x in enumerate(xcurr):
            e = mean_errors[n][i]
            y = aggregate_errors(all_errors[n][i], mode=AGGREGATION, eps=EPS)
            print(f"  -log2(mesh)={x:5.1f}   mean_error={e:.6e}   y={y:.6f}")
        print()


render_progress()

for n in DEGREES:
    for k in K_SWEEP:
        block_size = 2 ** k
        kernel = HigherOrderKernel(
            log_steps=(block_size, block_size),
            log_degree=(n, n),
            backend="scan",
            dyadic_order=DYO_HO,
            increment_input=True,
            num_devices=NUM_DEVICES,
        )
        approx_vals = np.asarray(
            kernel(
                dX,
                dY,
                evaluate="terminal",
                return_fg=False,
                pairwise=False,
            )
        )

        errs = np.abs(approx_vals - fine_vals)
        all_errors[n].append(errs)
        mean_errors[n].append(float(np.mean(errs)))

        render_progress()

From the figure in the higher-order paper:
| x = -log2(mesh) | deg 1 y | deg 2 y | deg 3 y | deg 4 y |
| --------------: | ------: | ------: | ------: | ------: |
|               1 |    0.85 |    1.45 |    2.25 |    3.00 |
|               2 |    0.95 |    1.60 |    2.55 |    3.35 |
|               3 |    1.10 |    1.70 |    2.75 |    4.05 |
|               4 |    1.10 |    2.00 |    3.25 |    4.25 |
|               5 |    1.25 |    2.25 |    3.75 |    4.90 |
|               6 |    1.35 |    2.65 |    4.00 |    5.60 |
|               7 |    1.55 |    2.90 |    4.60 |    6.20 |
|               8 |    1.55 |    3.30 |    5.00 |    6.65 |
